In [ ]:
# pip install fastapi uvicorn psycopg2-binary sqlalchemy faiss-cpu sentence-transformers

In [1]:
import pandas as pd

In [31]:
data = pd.read_csv('data.csv', encoding='utf-8')


In [32]:
data

,id_incident,date,description,priorité
0,INC00000001,2024-08-23,Le service informatique rencontrent une situat...,Moyenne
1,INC00000002,2021-03-04,Le service informatique rencontrent une situat...,Très faible
2,INC00000003,2022-11-05,Le service informatique rencontrent une situat...,Faible
3,INC00000004,2024-04-11,Le siège rencontrent une situation de bug visu...,Moyenne
4,INC00000005,2022-12-01,Un seul utilisateur rencontrent une situation ...,Faible
...,...,...,...,...
995,INC00000996,2021-01-29,Le siège rencontrent une situation de latence ...,Moyenne
996,INC00000997,2020-07-15,Une agence régionale rencontrent une situation...,Faible
997,INC00000998,2021-02-16,Un seul utilisateur rencontrent une situation ...,Moyenne
998,INC00000999,2022-01-14,Tous les utilisateurs rencontrent une situatio...,Critique


In [33]:
import re

def nettoyer_texte(texte):
    """Nettoyage simple : suppression des caractères spéciaux, mise en minuscule"""
    texte = texte.lower()
    texte = re.sub(r'[^\w\s]', '', texte)
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte


In [34]:
data['description_clean'] = data['description'].apply(nettoyer_texte)

In [35]:
data.head()

,id_incident,date,description,priorité,description_clean
0,INC00000001,2024-08-23,Le service informatique rencontrent une situat...,Moyenne,le service informatique rencontrent une situat...
1,INC00000002,2021-03-04,Le service informatique rencontrent une situat...,Très faible,le service informatique rencontrent une situat...
2,INC00000003,2022-11-05,Le service informatique rencontrent une situat...,Faible,le service informatique rencontrent une situat...
3,INC00000004,2024-04-11,Le siège rencontrent une situation de bug visu...,Moyenne,le siège rencontrent une situation de bug visu...
4,INC00000005,2022-12-01,Un seul utilisateur rencontrent une situation ...,Faible,un seul utilisateur rencontrent une situation ...


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Chargement du modèle BERT
modele_bert = SentenceTransformer("all-MiniLM-L6-v2")

def encoder_description(description):
    """Transforme une description en vecteur BERT"""
    description_propre = nettoyer_texte(description)
    vecteur = modele_bert.encode([description_propre])
    return vecteur[0].astype('float32')  # Format requis pour FAISS

In [36]:
# Encoder toutes les descriptions en une seule fois
vectors = modele_bert.encode(data['description_clean'].tolist())


In [37]:
import faiss

dimension = 384  # Taille des vecteurs BERT
index_faiss = faiss.IndexFlatL2(dimension)  

In [43]:
faiss_to_postgres = {}

In [ ]:
# Ajout de tous les vecteurs dans FAISS
index_faiss.add(np.array(vectors, dtype='float32'))


# Vérifier le nombre d'éléments indexés
print(f"Nombre d'éléments indexés dans FAISS : {index_faiss.ntotal}")


Nombre d'éléments indexés dans FAISS : 1000


In [42]:
query = "Problème de connexion à la base de données"
vec_query = encoder_description(query)
indices, distances = index_faiss.search(np.array([vec_query], dtype='float32'), k=5)

# Afficher les descriptions correspondantes
# session = SessionLocal()
# for idx in indices[0]:
#     incident = data.query(Incident).offset(idx).limit(1).first()
#     print(f"Incident #{incident.id}: {incident.description}")
# session.close()


#afficher descriptions correspondantes depuis data
for idx in indices[0]:
    idx = int(idx)  # Convertir l'index en entier
    print(f"Incident #{data.iloc[idx]['id_incident']}: {data.iloc[idx]['description']}")
    print(f"Distance: {distances[0][idx]}")
    print("-" * 40)


Incident #INC00000001: Le service informatique rencontrent une situation de panne critique sur le logiciel DBS.
Distance: 503
----------------------------------------
Incident #INC00000001: Le service informatique rencontrent une situation de panne critique sur le logiciel DBS.
Distance: 503
----------------------------------------
Incident #INC00000001: Le service informatique rencontrent une situation de panne critique sur le logiciel DBS.
Distance: 503
----------------------------------------
Incident #INC00000001: Le service informatique rencontrent une situation de panne critique sur le logiciel DBS.
Distance: 503
----------------------------------------
Incident #INC00000001: Le service informatique rencontrent une situation de panne critique sur le logiciel DBS.
Distance: 503
----------------------------------------


In [ ]:
# pourquoi les resultats sont-ils les meme?
# Les résultats sont les mêmes car nous avons utilisé le même modèle BERT pour encoder la requête et les descriptions.
# Les vecteurs encodés sont comparés dans l'index FAISS, qui est basé sur la distance euclidienne.
# Pour obtenir des résultats différents, il faudrait utiliser un modèle différent ou modifier la requête.
# utiliser un modèle différent ou modifier la requête pour obtenir des résultats différents.
# Pour utiliser un modèle différent, vous pouvez essayer d'autres modèles disponibles dans la bibliothèque `sentence-transformers`.
# Par exemple, vous pouvez remplacer "all-MiniLM-L6-v2" par un autre modèle comme "distilbert-base-nli-stsb-mean-tokens".
# Pour modifier la requête, vous pouvez changer le texte de la requête pour qu'il corresponde à un autre incident ou à une autre description.


In [ ]:
print(encoder_description("Ceci est un exemple de description."))

[-1.61952041e-02  6.17311411e-02  7.51508772e-03 -2.20188834e-02
 -1.99037092e-03  5.30362176e-03  9.10033733e-02  7.18366504e-02
  6.66384995e-02 -3.02938893e-02  3.50321531e-02 -1.66120321e-01
  3.84936780e-02 -4.80977632e-03 -1.33561166e-02 -1.07164592e-01
  4.68625799e-02  3.64167392e-02 -3.18408594e-03  4.66006882e-02
  6.65993094e-02 -2.44973842e-02 -2.09199376e-02  9.14924145e-02
 -7.25407749e-02  1.15802353e-02 -8.69287178e-04  2.97512170e-02
 -1.80446599e-02 -8.17423239e-02  7.07307756e-02  4.03873101e-02
  7.96118826e-02 -2.89324522e-02  6.59964606e-02 -5.47440071e-03
 -1.39213940e-02 -3.51571813e-02  4.22835313e-02 -6.25755936e-02
 -1.10586479e-01 -2.70376317e-02 -6.37628585e-02 -2.63885614e-02
  3.84081937e-02  8.12099595e-03  1.03284689e-02  1.86081696e-02
 -5.93955368e-02 -2.21525729e-02 -5.05293831e-02 -2.13018954e-02
  1.02079064e-02 -1.71145890e-02  1.63144488e-02  2.09495449e-03
 -6.13982975e-03 -6.21841587e-02  5.73591590e-02  2.70976732e-03
  4.71644215e-02 -7.37719

In [ ]:
import faiss

dimension = 384  # Taille des vecteurs BERT
index_faiss = faiss.IndexFlatL2(dimension)  

# Ajout des vecteurs à l'index
def ajouter_a_faiss(vecteur):
    index_faiss.add(np.array([vecteur], dtype='float32'))

# Recherche dans FAISS
def rechercher_similaires_faiss(vecteur_query, top_k=5):
    distances, indices = index_faiss.search(np.array([vecteur_query], dtype='float32'), top_k)
    return indices[0], distances[0]


In [19]:
ajouter_a_faiss(encoder_description("Ceci est un exemple de description."))
ajouter_a_faiss(encoder_description("Un autre exemple de description."))

In [20]:
print(rechercher_similaires_faiss(encoder_description("cette description est mieux"), top_k=1))

(array([0], dtype=int64), array([0.5238962], dtype=float32))


In [17]:
# taille de faiss
print("Taille de l'index FAISS :", index_faiss.ntotal)

Taille de l'index FAISS : 2
